# 04 — Evaluation & Model Comparison
Load trained models, run held-out evaluation, and compare
Random Forest vs. MLP vs. LSTM vs. Hybrid on all metrics.


## 1. Imports & setup

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from library.utils         import setup_logging, get_device
from library.config        import BatchConfig
from library.data          import load_data_rf
from library.features      import add_features, build_feature_matrix
from library.visualization import (plot_confusion, plot_per_fault_f1,
                                   plot_cross_location, plot_ablation)

setup_logging()
%matplotlib inline
plt.rcParams["figure.dpi"] = 120


## 2. Paths

In [ ]:
DATA_DIR    = "data"
RF_DIR      = "outputs/random_forest"
PT_DIR      = "outputs/sequence_models"
OUTPUT_DIR  = "outputs/evaluation"
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

cfg    = BatchConfig(DATA_DIR)
device = get_device()


## 3. Load evaluation data

In [ ]:
# Load a fresh hold-out sample for unbiased evaluation
df = load_data_rf(DATA_DIR, cfg, max_rows=500_000)
print(f"Evaluation set: {len(df):,} rows")


## 4. Evaluate Random Forest models

In [ ]:
import joblib
from sklearn.metrics import roc_auc_score, classification_report

def eval_rf_model(pkl_path, df, task="detection"):
    artifact = joblib.load(pkl_path)
    model    = artifact["model"]
    feats    = artifact["feature_names"]

    if task == "detection":
        X, _ = build_feature_matrix(df, feats)
        y    = df["fault_active"].values.astype(int)
        y_prob = model.predict_proba(X)[:, 1]
        y_pred = model.predict(X)
        auc    = roc_auc_score(y, y_prob)
        print(f"  AUC = {auc:.4f}")
        print(classification_report(y, y_pred,
              target_names=["Healthy", "Faulted"]))
        return auc, y, y_pred
    else:
        le   = artifact["label_encoder"]
        faulted = df[df["fault_active"]].copy()
        X, _ = build_feature_matrix(faulted, feats)
        y    = le.transform(faulted["fault_type"].values)
        y_prob = model.predict_proba(X)
        y_pred = model.predict(X)
        auc    = roc_auc_score(y, y_prob, multi_class="ovr", average="weighted")
        print(f"  Weighted OvR AUC = {auc:.4f}")
        print(classification_report(le.inverse_transform(y),
                                    le.inverse_transform(y_pred)))
        return auc, le.inverse_transform(y), le.inverse_transform(y_pred)

rf_det_path = pathlib.Path(RF_DIR) / "rf_model_detection.pkl"
rf_cls_path = pathlib.Path(RF_DIR) / "rf_model_classification.pkl"

results_summary = {}

if rf_det_path.exists():
    print("\nRF Detection:")
    auc_rf_det, y_rf, yp_rf = eval_rf_model(rf_det_path, df, "detection")
    results_summary["RF Detection AUC"] = auc_rf_det
    plot_confusion(y_rf, yp_rf, ["Healthy", "Faulted"],
                   "RF Detection", f"{OUTPUT_DIR}/confusion_rf_detection.png")


In [ ]:
if rf_cls_path.exists():
    print("\nRF Classification:")
    auc_rf_cls, y_rf_c, yp_rf_c = eval_rf_model(rf_cls_path, df, "classification")
    results_summary["RF Classification AUC"] = auc_rf_cls


## 5. Evaluate PyTorch models

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from soley_ml.models.neural_network import build_model
from soley_ml.models.trainer import evaluate_model

def eval_pytorch_model(pt_path, df, task="detection"):
    checkpoint  = torch.load(pt_path, map_location=device)
    mode        = checkpoint["mode"]
    n_features  = checkpoint["n_features"]
    n_classes   = checkpoint["n_classes"]
    class_names = checkpoint["class_names"]
    feat_names  = checkpoint["feature_names"]
    window_size = checkpoint["window_size"]

    model = build_model(
        n_features, n_classes, mode=mode,
        hidden_lstm=checkpoint["hidden_lstm"],
        hidden_mlp=checkpoint["hidden_mlp"],
        n_lstm_layers=checkpoint["n_lstm_layers"],
        dropout=checkpoint["dropout"],
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(device)

    if task == "detection":
        sub = df
        y   = sub["fault_active"].values.astype(int)
    else:
        sub = df[df["fault_active"]].copy()
        import joblib
        le  = joblib.load(pathlib.Path(PT_DIR) / "label_encoder_fault_classification.pkl")
        y   = le.transform(sub["fault_type"].values)

    X, _ = build_feature_matrix(sub, feat_names)
    X    = torch.tensor(X, dtype=torch.float32)

    # Replicate last step to form a fake window (quick eval — no DataLoader)
    X_win = X.unsqueeze(1).repeat(1, window_size, 1)   # (N, W, F)
    ds    = TensorDataset(X_win, torch.tensor(y, dtype=torch.long))
    loader = DataLoader(ds, batch_size=512, shuffle=False)

    y_true, y_pred, y_prob, report, auc = evaluate_model(
        model, loader, device, class_names=class_names
    )
    return auc, y_true, y_pred, class_names

pt_modes = ["mlp", "lstm", "hybrid"]
for mode in pt_modes:
    for task_tag, task in [("fault_detection", "detection"),
                           ("fault_classification", "classification")]:
        p = pathlib.Path(PT_DIR) / f"model_{task_tag}_{mode}.pt"
        if p.exists():
            print(f"\nPyTorch {mode.upper()} — {task}")
            try:
                auc, y_t, y_p, cnames = eval_pytorch_model(p, df, task)
                key = f"PT {mode.upper()} {task.capitalize()} AUC"
                results_summary[key] = auc
            except Exception as ex:
                print(f"  (skipped: {ex})")


## 6. Results comparison table

In [ ]:
summary_df = pd.DataFrame.from_dict(
    results_summary, orient="index", columns=["ROC AUC"]
).sort_values("ROC AUC", ascending=False)
display(summary_df.style.format("{:.4f}").bar(color="#2563eb"))
summary_df.to_csv(f"{OUTPUT_DIR}/model_comparison.csv")
print(f"Saved model_comparison.csv")


## 7. Reload & display previously saved figures

In [ ]:
from IPython.display import Image, display as ipy_disp

figure_dirs = [RF_DIR, PT_DIR]
for d in figure_dirs:
    pngs = sorted(pathlib.Path(d).glob("*.png"))
    for p in pngs[:4]:   # show first 4 per directory to keep notebook manageable
        print(f"\n{p.name}")
        ipy_disp(Image(str(p)))


## 8. Deployment model quick test

In [ ]:
dep_path = pathlib.Path(RF_DIR) / "rf_deployment_detection.pkl"
if dep_path.exists():
    print("Deployment model (SCADA + stress only):")
    auc_dep, y_dep, yp_dep = eval_rf_model(dep_path, df, "detection")
    results_summary["RF Deployment Detection AUC"] = auc_dep
    print(f"\nDeployment AUC: {auc_dep:.4f}")
    print("(This model works on real SCADA data — no device physics required)")
else:
    print("Deployment model not found — run notebook 02 first.")


## 9. Save final report

In [ ]:
lines = [
    "SOLEY ML — Model Comparison Report",
    "=" * 50, "",
    f"Data directory: {DATA_DIR}",
    f"Evaluation rows: {len(df):,}",
    "",
    "Model                                   ROC AUC",
    "-" * 50,
]
for k, v in sorted(results_summary.items(), key=lambda x: -x[1]):
    lines.append(f"  {k:<40s}  {v:.4f}")

report_text = "\n".join(lines)
report_path = pathlib.Path(OUTPUT_DIR) / "evaluation_report.txt"
report_path.write_text(report_text, encoding="utf-8")
print(report_text)
print(f"\nSaved to {report_path}")
